In [13]:
import os, math, random, json, time
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import pennylane as qml
from scipy.sparse import coo_matrix, diags
import matplotlib.pyplot as plt
import pandas as pd

# ==== GLOBAL CONFIG (fast profile) ====
GLOBAL_CFG = {
    "data_dir": "dataset/amazon-book",
    "save_dir": "./runs/balanced_cpu",

    # subset size
    "max_users": 6000,
    "max_pos_per_user": 10,
    "neg_per_pos": 1,
    "val_ratio": 0.10,

    # model capacity
    "d": 64,
    "K": 2,
    "q": 4,           # a bit lighter than GPU config
    "L": 2,

    # compute
    "backend": "lightning.qubit",  # CPU backend
    "micro_bs": 48,

    # training
    "epochs_lg": 8,
    "epochs_hyb": 6,
    "batch_size": 1536,            # keep modest for RAM
    "lr": 1.5e-3,
    "wd": 1e-6,
    "eval_every": 1,

    # quantum anneal
    "p_quantum_start": 0.4,
    "p_quantum_end": 1.0,
}

# Repro
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_default_dtype(torch.float32)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
print("Device:", device)

Device: cuda


In [14]:
def load_amazon_book_dir(data_dir="dataset/amazon-book"):
    """Read LightGCN-format train/test files ('u i1 i2 ...' or 'u i')."""
    def read_user_items(path):
        users, items = [], []
        with open(path, "r") as f:
            for line in f:
                toks = line.strip().split()
                if not toks: 
                    continue
                u = int(toks[0])
                if len(toks) == 2:
                    users.append(u); items.append(int(toks[1]))
                else:
                    for t in toks[1:]:
                        users.append(u); items.append(int(t))
        return np.array(users, dtype=np.int64), np.array(items, dtype=np.int64)

    train_path = os.path.join(data_dir, "train.txt")
    test_path  = os.path.join(data_dir, "test.txt")
    if not (os.path.exists(train_path) and os.path.exists(test_path)):
        raise FileNotFoundError(f"Missing train/test files in {data_dir}")

    u_train, i_train = read_user_items(train_path)
    u_test,  i_test  = read_user_items(test_path)

    n_users = int(max(u_train.max(initial=0), u_test.max(initial=0)) + 1)
    n_items = int(max(i_train.max(initial=0), i_test.max(initial=0)) + 1)
    print(f"Loaded {len(u_train)} train, {len(u_test)} test pairs. Users={n_users}, Items={n_items}")
    return (u_train, i_train), (u_test, i_test), n_users, n_items

In [15]:
def make_small_implicit_split(u_train, i_train, u_test, i_test, n_users, n_items,
                              max_users=30000, max_pos_per_user=30, neg_per_pos=1,
                              val_ratio=0.1, seed=SEED):
    rng = np.random.default_rng(seed)
    unique_users = np.unique(u_train)
    users_subset = rng.choice(unique_users, size=min(max_users, len(unique_users)), replace=False)

    # positives per user (cap)
    user_pos = {}
    for u, i in zip(u_train, i_train):
        if u in users_subset:
            lst = user_pos.setdefault(u, [])
            if len(lst) < max_pos_per_user:
                lst.append(int(i))

    pos_pairs = np.array([(u, i) for u, items in user_pos.items() for i in items], dtype=np.int64)
    y_pos = np.ones(len(pos_pairs), dtype=np.float32)

    # negatives (1 per positive)
    neg_pairs = []
    for (u, _) in pos_pairs:
        seen = set(user_pos.get(int(u), []))
        while True:
            j = int(rng.integers(0, n_items))
            if j not in seen:
                neg_pairs.append((u, j))
                break
    neg_pairs = np.array(neg_pairs, dtype=np.int64)
    y_neg = np.zeros(len(neg_pairs), dtype=np.float32)

    X = np.vstack([pos_pairs, neg_pairs])
    y = np.concatenate([y_pos, y_neg])

    Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=val_ratio, stratify=y, random_state=seed)
    print(f"Subset -> Train {len(Xtr)}, Val {len(Xva)}, PosRatio {ytr.mean():.3f}")
    return (Xtr, ytr), (Xva, yva)

class PairDataset(torch.utils.data.Dataset):
    def __init__(self, pairs, labels):
        self.u = torch.as_tensor(pairs[:, 0], dtype=torch.int64)
        self.i = torch.as_tensor(pairs[:, 1], dtype=torch.int64)
        self.y = torch.as_tensor(labels, dtype=torch.float32)
    def __len__(self): return len(self.u)
    def __getitem__(self, idx): return self.u[idx], self.i[idx], self.y[idx]

def make_loaders(train, val, batch_size=4096, num_workers=0):
    """Safe for notebooks/macOS (no multiprocessing)."""
    tr, va = PairDataset(*train), PairDataset(*val)
    loader_args = dict(pin_memory=(device.type == "cuda"), num_workers=num_workers, persistent_workers=False)
    train_loader = torch.utils.data.DataLoader(tr, batch_size=batch_size, shuffle=True,  **loader_args)
    val_loader   = torch.utils.data.DataLoader(va, batch_size=batch_size, shuffle=False, **loader_args)
    return train_loader, val_loader

def build_norm_adj_from_train_pairs(n_users, n_items, train_pairs):
    """Symmetric normalized adjacency for LightGCN propagation (positives only)."""
    N = n_users + n_items
    rows, cols = train_pairs[:, 0], train_pairs[:, 1] + n_users
    A_ui = coo_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(N, N))
    A = A_ui + A_ui.T
    deg = np.array(A.sum(axis=1)).flatten(); deg[deg == 0] = 1.0
    D_inv = diags(1.0 / np.sqrt(deg))
    return (D_inv @ A @ D_inv).tocsr()

class _SparseDenseMM(torch.autograd.Function):
    """Sparse @ dense in float32 — CUDA sparse does not support Half under autocast."""

    @staticmethod
    def forward(ctx, A: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        if not A.is_sparse:
            raise ValueError("_SparseDenseMM expects sparse A")
        ctx.save_for_backward(A, x)
        return torch.sparse.mm(A, x.float())

    @staticmethod
    def backward(ctx, grad_output):
        if grad_output is None:
            return None, None
        A, x = ctx.saved_tensors
        go = grad_output.float()
        At = A.transpose(0, 1).coalesce()
        grad_x = torch.sparse.mm(At, go)
        return None, grad_x.to(dtype=x.dtype, device=x.device)


def _sparse_mm_fp32_safe(A: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    return _SparseDenseMM.apply(A, x)


class LightGCNLite(nn.Module):
    def __init__(self, n_users, n_items, d=32, K=1, A_norm=None):
        super().__init__()
        self.n_users, self.n_items, self.K = n_users, n_items, K
        self.E = nn.Embedding(n_users + n_items, d)
        nn.init.normal_(self.E.weight, std=0.05)
        assert A_norm is not None
        self.A = self._to_torch_sparse(A_norm)

    @staticmethod
    def _to_torch_sparse(A_csr):
        A = A_csr.tocoo()
        idx = torch.tensor(np.vstack([A.row, A.col]), dtype=torch.long)
        val = torch.tensor(A.data, dtype=torch.float32)
        return torch.sparse_coo_tensor(idx, val, torch.Size(A.shape)).coalesce()

    def propagate(self):
        A = self.A.to(self.E.weight.device)
        w = self.E.weight
        embs = [w]
        x = w
        amp_off = torch.amp.autocast("cuda", enabled=False) if w.is_cuda else nullcontext()
        with amp_off:
            for _ in range(self.K):
                x = _sparse_mm_fp32_safe(A, x)
                embs.append(x)
            out = torch.stack(embs).mean(dim=0)
        return out

    def forward(self, u, i):
        all_emb = self.propagate()
        uemb = all_emb[u]; iemb = all_emb[self.n_users + i]
        return (uemb * iemb).sum(dim=-1)  # logits

In [16]:
class QuantumBlock(nn.Module):
    def __init__(self, q=3, L=1, in_dim=64, dev_name="lightning.qubit"):
        super().__init__()
        self.q, self.L = q, L
        self.proj = nn.Linear(in_dim, q)
        try:
            self.dev = qml.device(dev_name, wires=q)
        except Exception:
            print(f"[QuantumBlock] '{dev_name}' unavailable → using 'lightning.qubit'")
            self.dev = qml.device("lightning.qubit", wires=q)
        self.weights = nn.Parameter(torch.randn(L, q, 2, dtype=torch.float32) * 0.1)

        @qml.qnode(self.dev, interface="torch", diff_method="adjoint")
        def circuit(x, w):
            qml.templates.AngleEmbedding(x, wires=range(self.q), rotation="Y")
            for l in range(self.L):
                for wi in range(self.q):
                    qml.CNOT(wires=[wi, (wi + 1) % self.q])
                for wi in range(self.q):
                    qml.RX(w[l, wi, 0], wires=wi)
                    qml.RY(w[l, wi, 1], wires=wi)
            return [qml.expval(qml.PauliZ(wi)) for wi in range(self.q)]
        self.circuit = circuit

    def forward(self, x, micro_bs=32):
        xq = torch.tanh(self.proj(x)) * math.pi / 2.0  # [B,q]
        outs = []
        for s in range(0, xq.shape[0], micro_bs):
            xb = xq[s:s+micro_bs]
            for b in range(xb.shape[0]):
                out_b = self.circuit(xb[b], self.weights)  # list of q scalars
                outs.append(torch.stack(out_b).to(x.dtype))
        return torch.stack(outs, dim=0)  # [B,q]

class HybridQGNN(nn.Module):
    def __init__(self, n_users, n_items, d=32, K=1, A_norm=None,
                 q=3, L=1, p_quantum=1.0, dev_name="lightning.qubit"):
        super().__init__()
        self.encoder = LightGCNLite(n_users, n_items, d=d, K=K, A_norm=A_norm)
        self.quantum = QuantumBlock(q=q, L=L, in_dim=2*d, dev_name=dev_name)
        self.head = nn.Sequential(nn.Linear(q, 64), nn.ReLU(), nn.Linear(64, 1))
        self.fallback = nn.Linear(2*d, q)
        self.p_quantum = float(p_quantum)

    def set_p_quantum(self, p): self.p_quantum = float(p)

    def forward(self, u, i, micro_bs=32):
        all_emb = self.encoder.propagate()
        x = torch.cat([all_emb[u], all_emb[self.encoder.n_users + i]], dim=-1)
        # Encoder is fp32; AMP can make Linear / quantum outputs fp16 — align dtypes before masked writes.
        out_dtype = x.dtype
        if self.training and self.p_quantum < 1.0:
            mask = torch.rand(x.size(0), device=x.device) < self.p_quantum
            zq = torch.empty(x.size(0), self.quantum.q, device=x.device, dtype=out_dtype)
            if mask.any():
                zq[mask] = self.quantum(x[mask], micro_bs=micro_bs).to(out_dtype)
            if (~mask).any():
                zq[~mask] = self.fallback(x[~mask]).to(out_dtype)
        else:
            zq = self.quantum(x, micro_bs=micro_bs).to(out_dtype)
        return self.head(zq).squeeze(-1)  # logits

In [17]:
class MetricsLogger:
    def __init__(self, save_dir):
        self.rows = []; self.save_dir = Path(save_dir); self.save_dir.mkdir(parents=True, exist_ok=True)
    def log(self, **kv):
        self.rows.append({k: (float(v) if isinstance(v, (int, float)) else v) for k, v in kv.items()})
    def dataframe(self): return pd.DataFrame(self.rows)
    def save_csv(self, name="metrics.csv"):
        p = self.save_dir / name; self.dataframe().to_csv(p, index=False); return p
    def save_json(self, name="metrics.json"):
        p = self.save_dir / name; json.dump(self.rows, open(p, "w"), indent=2); return p

def save_ckpt(p, model, opt, extra):
    p = Path(p); p.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "extra": extra}, p)

def save_best(p, model, opt, metric_name, val, payload=None):
    payload = payload or {}; payload.update({"best_metric": float(val), "metric_name": metric_name})
    save_ckpt(p, model, opt, payload)

@dataclass
class TrainCfg:
    epochs_lg: int = GLOBAL_CFG["epochs_lg"]
    epochs_hyb: int = GLOBAL_CFG["epochs_hyb"]
    lr: float = GLOBAL_CFG["lr"]
    wd: float = GLOBAL_CFG["wd"]
    batch_size: int = GLOBAL_CFG["batch_size"]
    micro_bs: int = GLOBAL_CFG["micro_bs"]
    eval_every: int = GLOBAL_CFG["eval_every"]
    p_quantum_start: float = GLOBAL_CFG["p_quantum_start"]
    p_quantum_end: float = GLOBAL_CFG["p_quantum_end"]

# AMP scaler for CUDA
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

def _to_device(u, i, y):
    if device.type == "cuda":
        return (u.to(device, non_blocking=True),
                i.to(device, non_blocking=True),
                y.to(device, non_blocking=True))
    return u.to(device), i.to(device), y.to(device)

def train_epoch(model, loader, opt, micro_bs=32, desc="train",
                logger: MetricsLogger=None, epoch=None, model_name=None):
    model.train()
    crit = nn.BCEWithLogitsLoss()
    total = 0.0; n = 0; t0 = time.time()
    pbar = tqdm(loader, desc=desc, leave=False)
    for u, i, y in pbar:
        u, i, y = _to_device(u, i, y)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logit = model(u, i, micro_bs=micro_bs) if isinstance(model, HybridQGNN) else model(u, i)
            loss = crit(logit, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        total += loss.item() * u.size(0); n += u.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    dur = time.time() - t0; avg = total / max(1, n)
    if logger: logger.log(model=model_name, split="train", metric="BCE", epoch=epoch, value=avg, seconds=dur)
    return avg, dur

# ---- extra regression-style metrics ----
def eval_regression_metrics(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae  = np.mean(np.abs(y_true - y_pred))
    mse  = np.mean((y_true - y_pred)**2)
    rmse = np.sqrt(mse)
    # MAPE: exclude zero values to avoid division by near-zero (fixes bug with binary classification)
    non_zero_mask = np.abs(y_true) > eps
    if non_zero_mask.any():
        mape = np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100
    else:
        mape = np.nan  # All values are zero, MAPE is undefined
    wmape = 100 * np.sum(np.abs(y_true - y_pred)) / (np.sum(y_true) + eps)
    return dict(MAE=mae, MSE=mse, RMSE=rmse, MAPE=mape, WMAPE=wmape)

@torch.no_grad()
def eval_metrics(model, loader, micro_bs=32,
                 logger: MetricsLogger=None, epoch=None, model_name=None, split="val"):
    model.eval()
    ys, ps = [], []
    t0 = time.time()
    for u, i, y in loader:
        u, i, y = _to_device(u, i, y)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logit = model(u, i, micro_bs=micro_bs) if isinstance(model, HybridQGNN) else model(u, i)
            prob = torch.sigmoid(logit).cpu().numpy()
        ys.append(y.cpu().numpy()); ps.append(prob)

    if not ys:
        dur = time.time() - t0
        metrics = {"AUC": float("nan"), **{k: float("nan") for k in ["MAE","MSE","RMSE","MAPE","WMAPE"]}}
        if logger:
            for k, v in metrics.items():
                logger.log(model=model_name, split=split, metric=k, epoch=epoch, value=v, seconds=dur)
        return metrics, dur

    ys = np.concatenate(ys)
    ps = np.concatenate(ps)

    # main AUC
    try:
        auc = roc_auc_score(ys, ps)
    except Exception:
        auc = float(np.mean((ps > 0.5) == ys))

    reg = eval_regression_metrics(ys, ps)
    metrics = {"AUC": auc, **reg}
    dur = time.time() - t0

    if logger:
        for k, v in metrics.items():
            logger.log(model=model_name, split=split, metric=k, epoch=epoch, value=v, seconds=dur)
    return metrics, dur

C:\Users\dkaze\AppData\Local\Temp\ipykernel_5488\668787653.py:33: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


In [18]:
def main():
    args = type("obj", (object,), GLOBAL_CFG)()
    save_dir = Path(args.save_dir); save_dir.mkdir(parents=True, exist_ok=True)

    # ---------- DATA ----------
    (u_tr, i_tr), (u_te, i_te), n_users, n_items = load_amazon_book_dir(args.data_dir)
    (Xtr, ytr), (Xva, yva) = make_small_implicit_split(
        u_tr, i_tr, u_te, i_te, n_users, n_items,
        max_users=args.max_users, max_pos_per_user=args.max_pos_per_user,
        neg_per_pos=args.neg_per_pos, val_ratio=args.val_ratio, seed=SEED
    )
    # adjacency from positive train pairs only
    A_norm = build_norm_adj_from_train_pairs(n_users, n_items, Xtr[ytr == 1])

    # safe DataLoaders (no multiprocessing)
    train_loader, val_loader = make_loaders(
        (Xtr, ytr), (Xva, yva),
        batch_size=args.batch_size, num_workers=0
    )

    # ---------- METRICS ----------
    logger = MetricsLogger(save_dir)
    with open(save_dir / "run_config.json", "w") as f:
        json.dump(GLOBAL_CFG, f, indent=2)

    cfg = TrainCfg()

    # ---------- LightGCN ----------
    lg = LightGCNLite(n_users, n_items, d=args.d, K=args.K, A_norm=A_norm).to(device)
    opt_lg = torch.optim.Adam(lg.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    best_lg = {"auc": -1.0, "ep": 0}

    print("\nTraining LightGCN-lite …")
    for ep in range(1, cfg.epochs_lg + 1):
        tr_loss, _ = train_epoch(lg, train_loader, opt_lg,
                                 micro_bs=cfg.micro_bs,
                                 logger=logger, epoch=ep, model_name="LightGCN")
        m, _ = eval_metrics(lg, val_loader,
                            micro_bs=cfg.micro_bs,
                            logger=logger, epoch=ep, model_name="LightGCN", split="val")
        print(f"[LightGCN] ep {ep}/{cfg.epochs_lg} | loss {tr_loss:.4f} | "
              f"AUC {m['AUC']:.4f} | MAE {m['MAE']:.4f} | RMSE {m['RMSE']:.4f} | MAPE {m['MAPE']:.2f}%")

        save_ckpt(save_dir / f"lg_ep{ep}.pt", lg, opt_lg, {"epoch": ep, "val_auc": m["AUC"]})
        if m["AUC"] > best_lg["auc"]:
            best_lg.update({"auc": m["AUC"], "ep": ep})
            save_best(save_dir / "lg_best.pt", lg, opt_lg, "val_auc", m["AUC"], {"epoch": ep})

    # ---------- Hybrid QGNN ----------
    hyb = HybridQGNN(n_users, n_items, d=args.d, K=args.K, A_norm=A_norm,
                     q=args.q, L=args.L, p_quantum=cfg.p_quantum_start,
                     dev_name=args.backend).to(device)
    opt_hyb = torch.optim.Adam(hyb.parameters(), lr=cfg.lr * 0.7, weight_decay=cfg.wd)
    best_hyb = {"auc": -1.0, "ep": 0}

    print("\nTraining Hybrid QGNN …")
    # warmup with encoder frozen
    for p in hyb.encoder.parameters(): p.requires_grad = False
    warm_loss, _ = train_epoch(hyb, train_loader, opt_hyb,
                               micro_bs=cfg.micro_bs, logger=logger, epoch=0, model_name="HybridQGNN")
    warm_m, _ = eval_metrics(hyb, val_loader, micro_bs=cfg.micro_bs,
                             logger=logger, epoch=0, model_name="HybridQGNN", split="val")
    for p in hyb.encoder.parameters(): p.requires_grad = True
    print(f"[HybridQGNN] warmup | train BCE {warm_loss:.4f} | AUC {warm_m['AUC']:.4f}")

    # main epochs with p_quantum anneal
    for ep in range(1, cfg.epochs_hyb + 1):
        cur_p = cfg.p_quantum_start + (cfg.p_quantum_end - cfg.p_quantum_start) * (ep - 1) / max(1, cfg.epochs_hyb - 1)
        hyb.set_p_quantum(cur_p)

        tr_loss, _ = train_epoch(hyb, train_loader, opt_hyb,
                                 micro_bs=cfg.micro_bs,
                                 desc=f"Hybrid ep{ep}/{cfg.epochs_hyb} p_q={cur_p:.2f}",
                                 logger=logger, epoch=ep, model_name="HybridQGNN")
        m, _ = eval_metrics(hyb, val_loader, micro_bs=cfg.micro_bs,
                            logger=logger, epoch=ep, model_name="HybridQGNN", split="val")
        logger.log(model="HybridQGNN", split="meta", metric="p_quantum", epoch=ep, value=cur_p)

        print(f"[HybridQGNN] ep {ep}/{cfg.epochs_hyb} | p_q {cur_p:.2f} | "
              f"loss {tr_loss:.4f} | AUC {m['AUC']:.4f} | MAE {m['MAE']:.4f} | RMSE {m['RMSE']:.4f} | MAPE {m['MAPE']:.2f}%")

        save_ckpt(save_dir / f"hyb_ep{ep}.pt", hyb, opt_hyb, {"epoch": ep, "val_auc": m["AUC"], "p_quantum": cur_p})
        if m["AUC"] > best_hyb["auc"]:
            best_hyb.update({"auc": m["AUC"], "ep": ep})
            save_best(save_dir / "hyb_best.pt", hyb, opt_hyb, "val_auc", m["AUC"], {"epoch": ep, "p_quantum": cur_p})

    # ---------- Persist metrics & summary ----------
    csv_path = logger.save_csv("metrics.csv")
    json_path = logger.save_json("metrics.json")
    with open(save_dir / "summary.txt", "w") as f:
        f.write(
            f"LightGCN best val AUC: {best_lg['auc']:.4f} (epoch {best_lg['ep']})\n"
            f"HybridQGNN best val AUC: {best_hyb['auc']:.4f} (epoch {best_hyb['ep']})\n"
        )

    print("\nSaved:")
    print("  - per-epoch checkpoints: lg_ep*.pt, hyb_ep*.pt")
    print("  - best checkpoints     : lg_best.pt, hyb_best.pt")
    print(f"  - metrics CSV/JSON     : {csv_path.name}, {json_path.name}")
    print("  - summary              : summary.txt")

In [19]:
main()

Loaded 2380730 train, 603378 test pairs. Users=52643, Items=91599
Subset -> Train 108000, Val 12000, PosRatio 0.500

Training LightGCN-lite …


train:   0%|          | 0/71 [00:00<?, ?it/s]C:\Users\dkaze\AppData\Local\Temp\ipykernel_5488\668787653.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
C:\Users\dkaze\AppData\Local\Temp\ipykernel_5488\668787653.py:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


[LightGCN] ep 1/8 | loss 0.6865 | AUC 0.5510 | MAE 0.4997 | RMSE 0.4998 | MAPE 49.95%


train:   0%|          | 0/71 [00:00<?, ?it/s]C:\Users\dkaze\AppData\Local\Temp\ipykernel_5488\668787653.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
C:\Users\dkaze\AppData\Local\Temp\ipykernel_5488\668787653.py:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


[LightGCN] ep 2/8 | loss 0.6636 | AUC 0.5964 | MAE 0.4976 | RMSE 0.4977 | MAPE 49.50%


...

[LightGCN] ep 7/8 | loss 0.3211 | AUC 0.6578 | MAE 0.4624 | RMSE 0.4760 | MAPE 42.26%


train:   0%|          | 0/71 [00:00<?, ?it/s]C:\Users\dkaze\AppData\Local\Temp\ipykernel_5488\668787653.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
C:\Users\dkaze\AppData\Local\Temp\ipykernel_5488\668787653.py:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


[LightGCN] ep 8/8 | loss 0.2818 | AUC 0.6593 | MAE 0.4585 | RMSE 0.4746 | MAPE 41.49%

Training Hybrid QGNN …


train:   0%|          | 0/71 [00:00<?, ?it/s]C:\Users\dkaze\AppData\Local\Temp\ipykernel_5488\668787653.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


RuntimeError: Index put requires the source and destination dtypes match, got Float for the destination and Half for the source.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

save_dir = Path(GLOBAL_CFG["save_dir"])
metrics_csv = save_dir / "metrics.csv"

if not metrics_csv.exists():
    print(f"No metrics.csv found at {metrics_csv}")
else:
    df = pd.read_csv(metrics_csv)

    # Keep only validation metrics we care about
    want_metrics = ["AUC", "MAE", "MSE", "RMSE", "MAPE", "WMAPE"]
    val = df[(df["split"] == "val") & (df["metric"].isin(want_metrics))].copy()

    if val.empty:
        print("No validation metrics found in metrics.csv")
    else:
        # Find the epoch with BEST AUC per model
        best_rows = []
        for model, g in val.groupby("model"):
            auc_g = g[g["metric"] == "AUC"]
            if auc_g.empty:
                continue
            best_ep = auc_g.sort_values("value", ascending=False).iloc[0]["epoch"]
            # Take all wanted metrics at that epoch
            snap = g[g["epoch"] == best_ep].pivot_table(index="model", columns="metric", values="value", aggfunc="first")
            snap["epoch_best_auc"] = best_ep
            best_rows.append(snap)
        if not best_rows:
            print("Could not locate best AUC epochs per model.")
        else:
            best_table = pd.concat(best_rows).reset_index()

            # Reorder columns nicely
            cols = ["model", "epoch_best_auc"] + want_metrics
            for c in cols:
                if c not in best_table.columns:
                    best_table[c] = np.nan
            best_table = best_table[cols]

            # Round for display
            disp = best_table.copy()
            disp["epoch_best_auc"] = disp["epoch_best_auc"].astype(int)
            for c in want_metrics:
                disp[c] = disp[c].astype(float).round(6)

            # Build delta row: Hybrid - LightGCN (if both present)
            models_present = set(disp["model"].unique())
            if {"LightGCN", "HybridQGNN"}.issubset(models_present):
                lg = disp[disp["model"] == "LightGCN"].iloc[0]
                hy = disp[disp["model"] == "HybridQGNN"].iloc[0]
                delta = {"model": "Δ (Hybrid − LightGCN)", "epoch_best_auc": np.nan}
                for c in want_metrics:
                    delta[c] = float(hy[c]) - float(lg[c])
                delta_row = pd.DataFrame([delta])
                disp_comp = pd.concat([disp, delta_row], ignore_index=True)
            else:
                disp_comp = disp

            # Show the comparative table
            from IPython.display import display
            print("Best-AUC snapshot per model (validation split). Δ = Hybrid − LightGCN")
            display(disp_comp)

            # Also provide a per-epoch overview (wide) for quick eyeballing (optional)
            # One row per epoch per model, columns = metrics
            wide = (val.pivot_table(index=["model","epoch"], columns="metric", values="value", aggfunc="first")
                      .reset_index()
                      .sort_values(["model","epoch"]))
            print("Per-epoch validation metrics (wide view):")
            display(wide.head(20))  # show a preview

            # Save both tables
            disp_comp.to_csv(save_dir / "val_best_comparative.csv", index=False)
            wide.to_csv(save_dir / "val_metrics_per_epoch.csv", index=False)
            print("Saved:")
            print(" -", save_dir / "val_best_comparative.csv")
            print(" -", save_dir / "val_metrics_per_epoch.csv")

Best-AUC snapshot per model (validation split). Δ = Hybrid − LightGCN


,model,epoch_best_auc,AUC,MAE,MSE,RMSE,MAPE,WMAPE
0,HybridQGNN,2.0,0.694456,0.408147,0.235118,0.484890,1.623685e+09,81.629315
1,LightGCN,8.0,0.659237,0.458515,0.225291,0.474648,2.510728e+09,91.702930
2,Δ (Hybrid − LightGCN),NaN,0.035219,-0.050368,0.009827,0.010242,-8.870429e+08,-10.073615


Per-epoch validation metrics (wide view):


metric,model,epoch,AUC,MAE,MAPE,MSE,RMSE,WMAPE
0,HybridQGNN,0.0,0.509227,0.499998,2.509698e+09,0.250002,0.500002,99.999700
1,HybridQGNN,1.0,0.676036,0.498365,2.479529e+09,0.248396,0.498394,99.673092
2,HybridQGNN,2.0,0.694456,0.408147,1.623685e+09,0.235118,0.484890,81.629315
3,HybridQGNN,3.0,0.679067,0.378867,1.367988e+09,0.281480,0.530547,75.773375
4,HybridQGNN,4.0,0.670115,0.379425,1.341048e+09,0.297081,0.545051,75.885095
5,HybridQGNN,5.0,0.663900,0.382784,1.317860e+09,0.305459,0.552684,76.556897
6,HybridQGNN,6.0,0.659766,0.386550,1.341238e+09,0.311126,0.557786,77.310002
7,LightGCN,1.0,0.551047,0.499746,2.500210e+09,0.249751,0.499751,99.949247
8,LightGCN,2.0,0.596379,0.497617,2.500957e+09,0.247740,0.497735,99.523336
9,LightGCN,3.0,0.621610,0.490724,2.503289e+09,0.242223,0.492161,98.144756


Saved:
 - runs/balanced_cpu/val_best_comparative.csv
 - runs/balanced_cpu/val_metrics_per_epoch.csv
